# Project 30 — LongevityAI: Human Biological Age & Healthspan Intelligence

## NHANES population benchmarking + PhenoAge biological-aging analytics + transparent sensitivity engine

This project turns public human biomarker reference data into a **longevity / healthy-aging analytics system**.

### Business / research question
Can we build a reproducible system that:
1. benchmarks biological-aging signals against real human population reference data,
2. implements a published biological-age model,
3. explains which biomarkers mathematically drive the score,
4. separates **measurement and hypothesis generation** from unsupported “anti-aging cure” claims?

### What this notebook demonstrates
- real NHANES 2017–2018 human biomarker reference data
- biological-age modelling with the Levine PhenoAge framework
- unit-safe biomarker engineering
- population age/sex benchmarking
- model validation against NHANES reference summaries
- sensitivity analysis / explainability
- healthspan-oriented decision framing
- explicit scientific guardrails against causal overclaiming

> **Important:** This is a data-science portfolio project, not medical advice and not a diagnostic tool. A lower modelled biological age does not prove that an intervention extends lifespan.


## Data provenance

The reference table below is a compact extract of age/sex-stratified NHANES 2017–2018 biomarker summaries and PhenoAge summaries.

Primary sources:
- CDC NHANES 2017–2018 data portal: https://wwwn.cdc.gov/nchs/nhanes/search/DataPage.aspx?Cycle=2017-2018
- CDC laboratory overview: https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/overviewlab.aspx?BeginYear=2017
- Public NHANES reference implementation used for cross-checking population summaries: https://github.com/aadityageddam-ux/labage
- PhenoAge methodology: Levine et al. (2018), *An epigenetic biomarker of aging for lifespan and healthspan*.

The compact table contains **aggregate population statistics**, not identifiable participant-level records.


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
RANDOM_STATE = 42
print("Environment ready ✅")


## 1. Build the compact NHANES 2017–2018 reference table

In [ ]:
reference_records = [{"stratum":"male_18-29","age_band":"18-29","sex":"male","albumin_median":4.4,"creatinine_median":0.93,"glucose_median":99,"crp_median":1.17,"lymphocytePct_median":32.6,"mcv_median":87.8,"rdw_median":13,"alp_median":74,"wbc_median":6.7,"phenoage_median":19.73859019804786,"phenoage_n":201},{"stratum":"male_30-39","age_band":"30-39","sex":"male","albumin_median":4.3,"creatinine_median":0.94,"glucose_median":102,"crp_median":1.39,"lymphocytePct_median":31.8,"mcv_median":87.9,"rdw_median":13.3,"alp_median":70,"wbc_median":6.8,"phenoage_median":32.15885871750085,"phenoage_n":147},{"stratum":"male_40-49","age_band":"40-49","sex":"male","albumin_median":4.2,"creatinine_median":0.96,"glucose_median":105,"crp_median":1.83,"lymphocytePct_median":30.5,"mcv_median":89.3,"rdw_median":13.3,"alp_median":70,"wbc_median":6.6,"phenoage_median":41.66367874135007,"phenoage_n":163},{"stratum":"male_50-59","age_band":"50-59","sex":"male","albumin_median":4.1,"creatinine_median":0.98,"glucose_median":106,"crp_median":1.72,"lymphocytePct_median":31.1,"mcv_median":89.4,"rdw_median":13.6,"alp_median":74,"wbc_median":7,"phenoage_median":53.292730750704294,"phenoage_n":178},{"stratum":"male_60-69","age_band":"60-69","sex":"male","albumin_median":4,"creatinine_median":1,"glucose_median":114,"crp_median":1.94,"lymphocytePct_median":27.2,"mcv_median":90.2,"rdw_median":13.6,"alp_median":79,"wbc_median":7.3,"phenoage_median":64.84788790406819,"phenoage_n":222},{"stratum":"male_70-79","age_band":"70-79","sex":"male","albumin_median":4.1,"creatinine_median":0.97,"glucose_median":114,"crp_median":1.77,"lymphocytePct_median":27,"mcv_median":91,"rdw_median":13.8,"alp_median":74,"wbc_median":7,"phenoage_median":78.20344625969938,"phenoage_n":121},{"stratum":"male_80+","age_band":"80+","sex":"male","albumin_median":4,"creatinine_median":1.1,"glucose_median":111,"crp_median":1.39,"lymphocytePct_median":25.6,"mcv_median":92.6,"rdw_median":14.1,"alp_median":75,"wbc_median":7.2,"phenoage_median":None,"phenoage_n":0},{"stratum":"female_18-29","age_band":"18-29","sex":"female","albumin_median":4,"creatinine_median":0.69,"glucose_median":94,"crp_median":1.98,"lymphocytePct_median":30.5,"mcv_median":87.7,"rdw_median":13.3,"alp_median":67,"wbc_median":7.6,"phenoage_median":21.263370755865324,"phenoage_n":217},{"stratum":"female_30-39","age_band":"30-39","sex":"female","albumin_median":4,"creatinine_median":0.7,"glucose_median":97,"crp_median":2.01,"lymphocytePct_median":30.5,"mcv_median":89.1,"rdw_median":13.4,"alp_median":65,"wbc_median":7.3,"phenoage_median":32.16165733714601,"phenoage_n":199},{"stratum":"female_40-49","age_band":"40-49","sex":"female","albumin_median":4,"creatinine_median":0.72,"glucose_median":102,"crp_median":2.67,"lymphocytePct_median":30.2,"mcv_median":89.4,"rdw_median":13.4,"alp_median":71,"wbc_median":7.4,"phenoage_median":44.413795800842394,"phenoage_n":165},{"stratum":"female_50-59","age_band":"50-59","sex":"female","albumin_median":4,"creatinine_median":0.73,"glucose_median":102,"crp_median":2.49,"lymphocytePct_median":32.2,"mcv_median":89.5,"rdw_median":13.4,"alp_median":84,"wbc_median":7.2,"phenoage_median":50.792896959781984,"phenoage_n":216},{"stratum":"female_60-69","age_band":"60-69","sex":"female","albumin_median":4,"creatinine_median":0.79,"glucose_median":106,"crp_median":2.04,"lymphocytePct_median":30.9,"mcv_median":89.7,"rdw_median":13.5,"alp_median":81,"wbc_median":6.6,"phenoage_median":60.51494568080571,"phenoage_n":230},{"stratum":"female_70-79","age_band":"70-79","sex":"female","albumin_median":4,"creatinine_median":0.8,"glucose_median":107,"crp_median":2.2,"lymphocytePct_median":28.3,"mcv_median":90.9,"rdw_median":13.8,"alp_median":81,"wbc_median":7.1,"phenoage_median":73.42721376082501,"phenoage_n":117},{"stratum":"female_80+","age_band":"80+","sex":"female","albumin_median":3.9,"creatinine_median":0.91,"glucose_median":105,"crp_median":1.86,"lymphocytePct_median":23.7,"mcv_median":91.4,"rdw_median":13.9,"alp_median":77,"wbc_median":7.3,"phenoage_median":None,"phenoage_n":0}]
reference = pd.DataFrame(reference_records)

age_mid_map = {
    "18-29": 23.5,
    "30-39": 34.5,
    "40-49": 44.5,
    "50-59": 54.5,
    "60-69": 64.5,
    "70-79": 74.5,
    "80+": 82.0,
}
reference["age_midpoint"] = reference["age_band"].map(age_mid_map)
reference["approx_age_gap"] = reference["phenoage_median"] - reference["age_midpoint"]

print("Reference table shape:", reference.shape)
print("Age/sex strata:", reference["stratum"].nunique())
print("Observed PhenoAge reference N:", int(reference["phenoage_n"].sum()))
reference.head()


### Why the `80+` PhenoAge cells are missing
The public reference file does not provide PhenoAge summaries for the 80+ strata because the complete-panel fasting reference sample is not available there. We preserve those missing values rather than inventing them.


In [ ]:
audit = pd.DataFrame({
    "column": reference.columns,
    "dtype": [str(reference[c].dtype) for c in reference.columns],
    "missing": [int(reference[c].isna().sum()) for c in reference.columns],
})
print("Missingness audit")
display(audit)
assert reference["stratum"].is_unique
assert reference["phenoage_median"].notna().sum() == 12
print("Data integrity checks passed ✅")


## 2. Population biological-age trajectory

In [ ]:
plot_df = reference.dropna(subset=["phenoage_median"]).copy()
age_order = ["18-29", "30-39", "40-49", "50-59", "60-69", "70-79"]

fig, ax = plt.subplots(figsize=(10, 6))
for sex, g in plot_df.groupby("sex"):
    g = g.set_index("age_band").loc[age_order].reset_index()
    ax.plot(g["age_band"], g["phenoage_median"], marker="o", linewidth=2, label=sex.title())

ax.set_title("NHANES 2017–2018 Median PhenoAge by Age Band and Sex")
ax.set_xlabel("Chronological age band")
ax.set_ylabel("Median PhenoAge (years)")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


The trajectory should rise strongly with chronological age; the useful signal is not merely the raw PhenoAge value, but how an individual's value compares with an appropriate age/sex reference distribution.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for sex, g in plot_df.groupby("sex"):
    g = g.set_index("age_band").loc[age_order].reset_index()
    ax.plot(g["age_band"], g["approx_age_gap"], marker="o", linewidth=2, label=sex.title())

ax.axhline(0, linewidth=1)
ax.set_title("Approximate Median PhenoAge Gap by Population Stratum")
ax.set_xlabel("Age band")
ax.set_ylabel("Median PhenoAge − age-band midpoint (years)")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("Approximate average gap across observed strata:",
      round(plot_df["approx_age_gap"].mean(), 2), "years")
print("Note: midpoint gaps are descriptive approximations, not participant-level age acceleration.")


## 3. Biomarker trajectories across age

In [ ]:
biomarkers = [
    "albumin_median", "creatinine_median", "glucose_median", "crp_median",
    "lymphocytePct_median", "mcv_median", "rdw_median", "alp_median", "wbc_median"
]

trend = reference.groupby("age_band", sort=False)[biomarkers].mean().loc[
    ["18-29","30-39","40-49","50-59","60-69","70-79","80+"]
]
z = (trend - trend.mean()) / trend.std(ddof=0)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(z.T, aspect="auto")
ax.set_xticks(range(len(z.index)))
ax.set_xticklabels(z.index)
ax.set_yticks(range(len(biomarkers)))
ax.set_yticklabels([b.replace("_median","") for b in biomarkers])
ax.set_title("Standardised Biomarker Median Trends Across Age Bands")
ax.set_xlabel("Age band")
ax.set_ylabel("Biomarker")
fig.colorbar(im, ax=ax, label="Within-biomarker z-score")
plt.tight_layout()
plt.show()


## 4. Implement PhenoAge with explicit unit conversion

Inputs are accepted in familiar clinical units:
- albumin: g/dL
- creatinine: mg/dL
- glucose: mg/dL
- CRP: mg/L
- lymphocyte percentage: %
- MCV: fL
- RDW: %
- alkaline phosphatase: U/L
- WBC: K/µL
- age: years

The function converts units internally before applying the published linear predictor.


In [ ]:
PHENOAGE_PARAMETERS = {
    "intercept": -19.90667,
    "albumin": -0.03359355,
    "creatinine": 0.009506491,
    "glucose": 0.1953192,
    "crp_ln": 0.09536762,
    "lymphocyte_pct": -0.01199984,
    "mcv": 0.02676401,
    "rdw": 0.3306156,
    "alp": 0.001868778,
    "wbc": 0.05542406,
    "age": 0.08035356,
    "gompertz_numerator": 1.51714,
    "gompertz_shape": 0.007692696,
    "age_scale": 0.090165,
    "age_multiplier": 0.0055305,
    "age_intercept": 141.50225,
}

def compute_phenoage(age, albumin, creatinine, glucose, crp,
                     lymphocytePct, mcv, rdw, alp, wbc):
    if crp <= 0:
        raise ValueError("CRP must be > 0 because the model uses log(CRP).")
    p = PHENOAGE_PARAMETERS
    albumin_g_L = albumin * 10.0
    creatinine_umol_L = creatinine * 88.4017
    glucose_mmol_L = glucose * 0.0555
    crp_mg_dL = crp * 0.1

    xb = (
        p["intercept"]
        + p["albumin"] * albumin_g_L
        + p["creatinine"] * creatinine_umol_L
        + p["glucose"] * glucose_mmol_L
        + p["crp_ln"] * np.log(crp_mg_dL)
        + p["lymphocyte_pct"] * lymphocytePct
        + p["mcv"] * mcv
        + p["rdw"] * rdw
        + p["alp"] * alp
        + p["wbc"] * wbc
        + p["age"] * age
    )

    log_hazard = np.log(p["gompertz_numerator"] / p["gompertz_shape"]) + xb
    phenoage = p["age_intercept"] + (
        np.log(p["age_multiplier"]) + log_hazard
    ) / p["age_scale"]
    return float(phenoage)


## 5. Cross-check the implementation against the real NHANES reference summaries

In [ ]:
def row_profile_phenoage(row):
    return compute_phenoage(
        age=row["age_midpoint"],
        albumin=row["albumin_median"],
        creatinine=row["creatinine_median"],
        glucose=row["glucose_median"],
        crp=row["crp_median"],
        lymphocytePct=row["lymphocytePct_median"],
        mcv=row["mcv_median"],
        rdw=row["rdw_median"],
        alp=row["alp_median"],
        wbc=row["wbc_median"],
    )

validation = reference.dropna(subset=["phenoage_median"]).copy()
validation["median_profile_phenoage"] = validation.apply(row_profile_phenoage, axis=1)
validation["difference"] = validation["median_profile_phenoage"] - validation["phenoage_median"]

mae = mean_absolute_error(validation["phenoage_median"], validation["median_profile_phenoage"])
corr = validation[["phenoage_median", "median_profile_phenoage"]].corr().iloc[0, 1]

print(f"Median-profile approximation MAE: {mae:.2f} years")
print(f"Correlation with reported stratum medians: {corr:.4f}")
display(validation[["stratum", "phenoage_median", "median_profile_phenoage", "difference"]].round(2))


**Interpretation:** a nonlinear model applied to stratum medians does not mathematically equal the median of participant-level outputs. The high correlation is a trajectory cross-check; residual differences are expected aggregation effects.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(validation["phenoage_median"], validation["median_profile_phenoage"], s=70)
lo = min(validation["phenoage_median"].min(), validation["median_profile_phenoage"].min())
hi = max(validation["phenoage_median"].max(), validation["median_profile_phenoage"].max())
ax.plot([lo, hi], [lo, hi], linestyle="--")
ax.set_title("Formula Cross-Check Against NHANES Stratum Medians")
ax.set_xlabel("Reported median PhenoAge")
ax.set_ylabel("Computed median-profile PhenoAge")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 6. Explainability: one-at-a-time sensitivity analysis

For a representative **45-year-old** profile using the male 40–49 NHANES median biomarkers, perturb one biomarker at a time by ±10%.

This answers **how sensitive the mathematical score is to an input**. It does not prove that changing that biomarker by treatment extends lifespan.


In [ ]:
base_row = reference.loc[reference["stratum"] == "male_40-49"].iloc[0]
base_profile = {
    "age": 45.0,
    "albumin": base_row["albumin_median"],
    "creatinine": base_row["creatinine_median"],
    "glucose": base_row["glucose_median"],
    "crp": base_row["crp_median"],
    "lymphocytePct": base_row["lymphocytePct_median"],
    "mcv": base_row["mcv_median"],
    "rdw": base_row["rdw_median"],
    "alp": base_row["alp_median"],
    "wbc": base_row["wbc_median"],
}
base_age = compute_phenoage(**base_profile)
print(f"Reference-profile PhenoAge: {base_age:.2f} years")
print(f"Reference-profile age gap: {base_age - base_profile['age']:+.2f} years")


In [ ]:
sensitivity_rows = []
for biomarker in ["albumin","creatinine","glucose","crp","lymphocytePct","mcv","rdw","alp","wbc"]:
    lower = base_profile.copy()
    upper = base_profile.copy()
    lower[biomarker] *= 0.90
    upper[biomarker] *= 1.10
    lower_age = compute_phenoage(**lower)
    upper_age = compute_phenoage(**upper)
    sensitivity_rows.append({
        "biomarker": biomarker,
        "minus_10pct_delta_years": lower_age - base_age,
        "plus_10pct_delta_years": upper_age - base_age,
        "max_abs_delta": max(abs(lower_age - base_age), abs(upper_age - base_age))
    })

sensitivity = pd.DataFrame(sensitivity_rows).sort_values("max_abs_delta", ascending=False)
display(sensitivity.round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ordered = sensitivity.sort_values("plus_10pct_delta_years")
ax.barh(ordered["biomarker"], ordered["plus_10pct_delta_years"])
ax.axvline(0, linewidth=1)
ax.set_title("PhenoAge Model Sensitivity to a +10% Biomarker Perturbation")
ax.set_xlabel("Change in modelled PhenoAge (years)")
ax.set_ylabel("Biomarker")
plt.tight_layout()
plt.show()


## 7. Population sample coverage

A useful longevity model should show where evidence is strong and where reference data are thinner.


In [ ]:
coverage = (
    reference.dropna(subset=["phenoage_median"])
    .pivot(index="age_band", columns="sex", values="phenoage_n")
    .loc[age_order]
)

fig, ax = plt.subplots(figsize=(10, 6))
coverage.plot(kind="bar", ax=ax)
ax.set_title("NHANES Reference Sample Counts Used for PhenoAge Summaries")
ax.set_xlabel("Age band")
ax.set_ylabel("Reference participants with complete PhenoAge panel")
ax.legend(title="Sex")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("Total complete-panel reference observations represented:", int(reference["phenoage_n"].sum()))


## 8. Human anti-aging / longevity research decision layer

A credible system for evaluating a **human longevity intervention** should go beyond a single biological-age score.

### Minimum evidence stack
1. **Baseline + repeated measures** — establish within-person change.
2. **Comparator / control group** — distinguish intervention effect from regression to the mean.
3. **Pre-registered outcomes** — avoid cherry-picking biomarkers.
4. **Multiple aging dimensions** — clinical chemistry, function, wearables, cognition, potentially omics.
5. **Safety endpoints** — improvement in one score must not hide harm elsewhere.
6. **Causal design** — randomisation when feasible, otherwise carefully justified causal inference.
7. **Long-term outcomes** — healthspan, disease incidence and ultimately survival matter more than short-term score movement.

### Product concept
`LongevityAI` can operate as a **measurement + evidence platform**:
- ingest biomarker panels,
- calculate validated aging scores,
- benchmark against population references,
- track longitudinal change,
- detect unusual shifts,
- compare intervention cohorts,
- generate transparent evidence reports.

The model should support scientists and clinicians — not replace them.


## 9. Final solution

In [ ]:
observed = reference.dropna(subset=["phenoage_median"])
top_sens = sensitivity.iloc[0]

print("PROJECT 30 — LONGEVITYAI FINAL RESULTS")
print("=" * 76)
print("Dataset: NHANES 2017–2018 age/sex population biomarker reference")
print(f"Reference strata analysed: {len(reference)}")
print(f"Complete-panel PhenoAge observations represented: {int(reference['phenoage_n'].sum()):,}")
print()
print("BIOLOGICAL-AGE ENGINE")
print("-" * 76)
print(f"Formula cross-check correlation: {corr:.4f}")
print(f"Median-profile approximation MAE: {mae:.2f} years")
print(f"Representative 45-year-old median-profile PhenoAge: {base_age:.2f} years")
print(f"Representative age gap: {base_age - 45:+.2f} years")
print()
print("EXPLAINABILITY")
print("-" * 76)
print(f"Largest ±10% mathematical sensitivity in this profile: {top_sens['biomarker']} "
      f"(max absolute shift {top_sens['max_abs_delta']:.2f} years)")
print()
print("SOLUTION")
print("-" * 76)
print("Use LongevityAI as a population-benchmarked biological-aging measurement")
print("and longitudinal research platform. Treat sensitivity outputs as model")
print("explanations, not proof that manipulating a biomarker reverses human aging.")
print()
print("PROJECT 30 COMPLETE ✅")
